In [ ]:
import pandas as pd
import requests
import time
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# CONFIGURATION
# ==========================================
TMDB_API_KEY = "92a48f2ce3dea72aecb5346f1c502b27"
INPUT_CSV = "/Users/adonisgeoffmacias/Documents/GitHub/data-trio-project/DMW/Data Scrapping RESULTS/user_ratings.csv"
OUTPUT_CSV = "user_ratings_enriched2.csv"

# Adjust this based on TMDB limits (TMDB allows ~40-50 requests/sec). 
# 10 workers is generally a safe, fast sweet spot.
MAX_WORKERS = 10 

# Using a session speeds up requests by reusing underlying TCP connections
session = requests.Session()

# ==========================================
# FUNCTIONS
# ==========================================
def get_genre_mapping(api_key):
    """Fetches the official TMDB list of movie genres to map IDs to names."""
    url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={api_key}&language=en-US"
    response = session.get(url)
    if response.status_code == 200:
        genres = response.json().get('genres', [])
        return {g['id']: g['name'] for g in genres}
    print("Warning: Could not fetch genre mapping from TMDB.")
    return {}

def fetch_movie_data(slug, api_key, genre_mapping):
    """Searches TMDB for a movie based on the Letterboxd slug."""
    match = re.search(r'-(\d{4})$', str(slug))
    year = match.group(1) if match else None
    
    clean_title = re.sub(r'-\d{4}$', '', str(slug)).replace('-', ' ')
    
    url = f"https://api.themoviedb.org/3/search/movie?api_key={api_key}&query={clean_title}"
    if year:
        url += f"&primary_release_year={year}"
        
    # Implement basic retry logic in case we hit the API rate limit (HTTP 429)
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = session.get(url)
            
            # If we hit the rate limit, sleep and try again
            if response.status_code == 429:
                time.sleep(1 * (attempt + 1))
                continue
                
            if response.status_code == 200:
                results = response.json().get('results', [])
                if results:
                    movie = results[0]
                    
                    genre_names = [genre_mapping.get(gid) for gid in movie.get('genre_ids', []) if gid in genre_mapping]
                    
                    return {
                        'film_slug': slug,
                        'tmdb_id': movie.get('id'),
                        'proper_title': movie.get('title'),
                        'genres_list': genre_names
                    }
            break # Break out of retry loop if successful or it's a hard error (like 404)
            
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Error fetching data for slug '{slug}': {e}")
            time.sleep(0.5)
            
    # Return empty/null values if not found or if there's an error
    return {
        'film_slug': slug,
        'tmdb_id': None,
        'proper_title': None,
        'genres_list': []
    }

# ==========================================
# MAIN SCRIPT
# ==========================================
def main():
    print(f"Loading data from {INPUT_CSV}...")
    try:
        df = pd.read_csv(INPUT_CSV)
    except FileNotFoundError:
        print(f"Error: Could not find file at {INPUT_CSV}")
        return

    unique_slugs = df['film_slug'].dropna().unique()
    print(f"Found {len(unique_slugs)} unique movies to fetch.")
    
    print("Fetching TMDB genre list...")
    genre_mapping = get_genre_mapping(TMDB_API_KEY)
    
    movie_details = []
    
    print(f"Fetching movie details from TMDB using {MAX_WORKERS} concurrent threads...")
    
    # --- MULTITHREADING IMPLEMENTATION ---
    # We use ThreadPoolExecutor to run multiple API requests simultaneously
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks to the executor
        future_to_slug = {executor.submit(fetch_movie_data, slug, TMDB_API_KEY, genre_mapping): slug for slug in unique_slugs}
        
        # Use tqdm to create a progress bar as the futures complete
        for future in tqdm(as_completed(future_to_slug), total=len(unique_slugs), desc="Fetching TMDB Data", unit="movie"):
            try:
                data = future.result()
                movie_details.append(data)
            except Exception as exc:
                slug = future_to_slug[future]
                print(f"{slug} generated an exception: {exc}")

    details_df = pd.DataFrame(movie_details)
    
    print("Formatting genres into separate columns...")
    for i in range(5):
        col_name = f'genre_{i+1}'
        details_df[col_name] = details_df['genres_list'].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        
    details_df = details_df.drop(columns=['genres_list'])
    
    print("Merging data back together...")
    final_df = pd.merge(df, details_df, on='film_slug', how='left')
    
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Success! Data has been saved to {OUTPUT_CSV}")

if _name_ == "_main_":
    main()